<div style="font-family: Arial, sans-serif; background-color: #f8f9fa; padding: 20px; border-radius: 10px;">
  <img src="https://sigmoidal.ai/wp-content/uploads/2024/09/Academia-Sigmoidal-Light.png" alt="Academia Sigmoidal" width="250">
  <h1 style="color: #007bff; font-size: 24px;">Bootcamp de Visão Computacional com Drones</h1>
  <h3 style="color: #343a40; font-size: 20px;">Projeto 4: Adaptar o detector às imagens aéreas</h3>
  <p style="color: #6c757d; font-size: 14px;"><strong>Instrutor:</strong> Carlos Melo, MSc.</p>
</div>


# Adaptar o detector às imagens aéreas

Vamos ajustar o YOLO11n com o treino, selecionar o checkpoint pela validação e avaliar a contagem com o mesmo protocolo do baseline. O treinamento curto é uma hipótese experimental; uma melhoria não está garantida.

Este notebook é independente: os dados necessários acompanham o repositório. No Colab, selecione **Copiar para o Drive** e execute as células na ordem. Selecione uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlosfab/bootcamp-visao-computacional-drones/blob/codex/projeto-4-gado/projeto-4/02_fine_tuning.ipynb)


## Ambiente

A primeira célula obtém os arquivos da versão do projeto e instala as dependências. O PyTorch fornecido pelo ambiente será preservado; sua versão será registrada. Se já importou bibliotecas antes desta instalação, reinicie o ambiente e execute novamente desde o início.

Os arquivos de trabalho ficam em `dados/`, `pesos/` e `resultados/`. Antes de encerrar o Colab, salve seus resultados no Drive ou faça download da pasta correspondente.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
if not Path("suporte").is_dir():
    destino = Path("/content/bootcamp-gado")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", "codex/projeto-4-gado",
                        "https://github.com/carlosfab/bootcamp-visao-computacional-drones.git", str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "sparse-checkout", "set", "projeto-4"], check=True)
    os.chdir(destino / "projeto-4")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from suporte.preparar_dados import obter_dados
from suporte.experimento import inventario, ler_caixas, desenhar
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 12,
                     "font.family": "sans-serif", "axes.grid": False})


A semente controla parte da aleatoriedade, mas versões, hardware e operações numéricas também influenciam o resultado. O registro abaixo identifica o ambiente desta execução, sem prometer identidade bit a bit entre CPU e GPU.


In [ ]:
import torch
from ultralytics import YOLO
from importlib.metadata import version
from suporte.experimento import inferir, calibrar, salvar_resultado
from suporte.contagem import evaluate_predictions
DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO)


In [ ]:
SAIDA = Path("resultados/fine_tuning")
SAIDA.mkdir(parents=True, exist_ok=True)
pacotes = ["numpy", "pandas", "matplotlib", "Pillow", "ultralytics", "torch"]
ambiente = {nome: version(nome) for nome in pacotes}
ambiente["python"] = platform.python_version()
ambiente["sistema"] = platform.platform()
ambiente["dispositivo"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
(SAIDA / "ambiente.json").write_text(json.dumps(ambiente, indent=2))
display(ambiente)


## Dados e partições

O recorte didático contém **300 imagens** de bovinos a pasto: 180 de treino, 60 de validação e 60 de teste. As imagens derivam do ICAERUS v2, com lado maior limitado a 2048 pixels e anotações YOLO de uma classe, `cow`. A atribuição e a licença CC BY 4.0 acompanham os dados.

Treino e validação usam Mauron e Jalogny em datas separadas. O teste usa Derval, uma fazenda ausente do desenvolvimento. Datas e voos não atravessam partições, mas os animais não têm identificadores individuais. A avaliação mede contagem **por imagem**, não um censo de indivíduos únicos.


In [ ]:
RAIZ = obter_dados()
tabela = inventario(RAIZ)
manifesto = json.loads((RAIZ / "manifesto.json").read_text())
treino = tabela[tabela["split"] == "train"].copy()
validacao = tabela[tabela["split"] == "val"].copy()
teste = tabela[tabela["split"] == "test"].copy()
assert [len(treino), len(validacao), len(teste)] == [180, 60, 60]
print("Imagens de treino, validação e teste:", len(treino), len(validacao), len(teste))


## Configuração antes do treinamento

Usaremos 25 épocas, batch de 8 imagens, entrada de 640 pixels e semente 42. A GPU é recomendada; a CPU pode executar, mas seu tempo não representa o percurso planejado para aula. Não instalamos outra versão de PyTorch sobre a fornecida pelo Colab.

O diretório de treino deve ser novo. Para investigar outra configuração, escolha outro nome e registre a mudança antes de consultar o teste. A semente não garante resultados idênticos entre diferentes ambientes.


In [ ]:
modelo = YOLO("yolo11n.pt")
cfg = dict(data=str(RAIZ.relative_to(Path.cwd()) / "data.yaml"), epochs=25, batch=8, imgsz=640,
           seed=42, workers=2, deterministic=True, cache=False, plots=False,
           device=DISPOSITIVO, project="resultados/treino", name="gado", exist_ok=False)
assert not Path(cfg["project"], cfg["name"]).exists(), "Use outro nome para uma nova execução."
(SAIDA / "configuracao_treino.json").write_text(json.dumps(cfg, indent=2))
display(cfg)


**Fine-tuning** parte dos pesos genéricos e os atualiza com as imagens do domínio. A validação acompanha o treinamento e determina `best.pt`; o teste não participa da seleção. Os argumentos completos usados pela biblioteca também ficam registrados em `args.yaml` dentro da execução.

Não interrompa a célula para escolher uma época que parece boa no teste. Se o runtime cair, trate os arquivos existentes como um treino possivelmente incompleto e preserve-os para diagnóstico.


In [ ]:
from time import perf_counter
inicio = perf_counter()
modelo.train(**cfg)
segundos_treino = perf_counter() - inicio
pasta_treino = Path(modelo.trainer.save_dir)
melhor = Path(modelo.trainer.best)
assert melhor.is_file(), "O treinamento não produziu best.pt."
print("Treino concluído em segundos:", round(segundos_treino, 1))
print("Checkpoint:", melhor)


## Ler as curvas de treinamento

A perda de localização de treino e a de validação ajudam a investigar o ajuste. Queda da perda de treino sem melhora na validação pode indicar que o modelo está se adaptando demais à amostra. As curvas não autorizam escolher parâmetros pelo teste.


In [ ]:
historico = pd.read_csv(pasta_treino / "results.csv")
historico.columns = historico.columns.str.strip()
plt.plot(historico["epoch"], historico["train/box_loss"], label="Treino")
plt.plot(historico["epoch"], historico["val/box_loss"], label="Validação")
plt.xlabel("Época")
plt.ylabel("Perda de localização")
plt.legend()
plt.show()


O registro de conclusão associa o checkpoint aos bytes dos pesos e ao ambiente. Essa identidade permite distinguir seu treinamento de um peso de referência distribuído com o projeto. O arquivo só é gravado depois do retorno normal de `train()`.


In [ ]:
conclusao = {"weights": str(melhor.resolve().relative_to(Path.cwd())), "weights_sha256": hashlib.sha256(melhor.read_bytes()).hexdigest(),
             "epochs_completed": len(historico), "seconds": segundos_treino,
             "checkpoint_selection": "best.pt selecionado pela validação durante train()",
             "initial_weights_sha256": hashlib.sha256(Path("yolo11n.pt").read_bytes()).hexdigest(),
             "manifest_sha256": hashlib.sha256((RAIZ / "manifesto.json").read_bytes()).hexdigest()}
(SAIDA / "treinamento_concluido.json").write_text(json.dumps(conclusao, indent=2))
display(conclusao)
ajustado = YOLO(str(melhor))
display(ajustado.names)


## Calibrar o modelo ajustado

A escala das confianças pode mudar após o treinamento. Escolheremos o limiar novamente na validação, pela mesma regra do baseline: maior F1; em empate, menor MAE e maior limiar. Entrada e regra de associação permanecem iguais.


In [ ]:
previsoes_val = inferir(ajustado, validacao, device=DISPOSITIVO)
limiar, calibracao = calibrar(previsoes_val)
calibracao.to_csv(SAIDA / "calibracao.csv", index=False)
display(calibracao[["threshold", "precision", "recall", "f1", "mae"]])
print("Limiar congelado:", limiar)
salvar_resultado(SAIDA / "val", previsoes_val, limiar)


Salvamos a configuração de inferência antes da avaliação final. Reutilizar o teste entre os métodos planejados permite comparação direta; mudar o treinamento depois de examinar esse teste seria um novo ciclo de desenvolvimento.


In [ ]:
avaliacao = {"weights_sha256": conclusao["weights_sha256"], "conf": limiar,
             "imgsz": 640, "iou_match": 0.5, "tiled": False,
             "selecao": "F1 val; empate por menor MAE e maior confiança"}
(SAIDA / "configuracao_avaliacao.json").write_text(json.dumps(avaliacao, indent=2))
previsoes_teste = inferir(ajustado, teste, device=DISPOSITIVO)
resumo = salvar_resultado(SAIDA / "test", previsoes_teste, limiar)
chaves = ["n_images", "precision", "recall", "f1", "mae", "rmse", "bias", "exact_accuracy"]
display(pd.Series({chave: resumo[chave] for chave in chaves}, name="Fine-tuning"))
display(resumo["by_presence"])


## Comparar com a referência genérica

Se o baseline foi executado neste mesmo diretório, a tabela abaixo reúne seus resultados. Caso contrário, o notebook continua independente e apresenta apenas o treino atual; execute o notebook do baseline e reúna os arquivos ao preparar a entrega.

Compare as mesmas imagens, versões e regras. Um F1 maior não implica necessariamente um MAE menor, pois omissões e falsos positivos podem se compensar na contagem.


In [ ]:
comparacao = {"fine_tuning": {chave: resumo[chave] for chave in chaves}}
arquivo_baseline = Path("resultados/baseline/test/resumo.json")
if arquivo_baseline.exists():
    cfg_baseline = json.loads(Path("resultados/baseline/configuracao.json").read_text())
    assert cfg_baseline["manifest_sha256"] == conclusao["manifest_sha256"], "Baseline usa outros dados."
    baseline = json.loads(arquivo_baseline.read_text())
    comparacao["baseline"] = {chave: baseline[chave] for chave in chaves}
tabela_comparacao = pd.DataFrame.from_dict(comparacao, orient="index")
tabela_comparacao.to_csv(SAIDA / "comparacao.csv")
display(tabela_comparacao)


## Examinar erros concretos

A galeria seleciona as imagens com mais falsos positivos e falsos negativos, usando o limiar já congelado. **Verde** representa a anotação; **vermelho tracejado**, a previsão. Uma contagem correta pode esconder uma omissão compensada por uma caixa falsa.

Depois desta inspeção, registre hipóteses de falha. Uma alteração motivada pelo teste passa a ser exploração e exigiria outro conjunto independente para uma nova avaliação final.


In [ ]:
por_imagem = pd.read_csv(SAIDA / "test" / "per_image.csv")
por_imagem["falhas"] = por_imagem["fp"] + por_imagem["fn"]
casos = por_imagem.sort_values(["falhas", "image_id"], ascending=[False, True]).head(4)
display(casos[["image_id", "gt_count", "pred_count", "fp", "fn"]])
indice = {linha["image_id"]: linha for linha in previsoes_teste}
(SAIDA / "galeria").mkdir(exist_ok=True)


In [ ]:
for numero, image_id in enumerate(casos["image_id"]):
    linha = indice[image_id]
    figura = desenhar(linha["image"], linha["gt_boxes"], linha["pred_boxes"],
                      linha["scores"], limiar)
    figura.savefig(SAIDA / "galeria" / f"caso_{numero}.png", dpi=120)
    plt.show()
    plt.close(figura)


## Interpretar o ajuste

Identifique o que mudou e o que permaneceu difícil. Distinga o efeito sobre imagens positivas e negativas, e explique por que sua recomendação pode favorecer o baseline mesmo depois de um treinamento válido.

Preserve `best.pt`, `args.yaml`, `results.csv`, a configuração e o registro de ambiente. No próximo notebook, um checkpoint fixado permitirá estudar o efeito dos recortes sem depender de um treinamento anterior neste runtime.
